# Lekcija 03 - Agentni obrasci dizajna

U ovoj lekciji proučavamo tri temeljna obrasca dizajna za izgradnju učinkovitih AI agenata:

1. **Jasne upute za agente** — Izrada preciznih, uloge definirajućih upita koji usmjeravaju ponašanje agenata  
2. **Strukturirani izlaz s Pydantic modelima** — Osiguravanje da agenti vraćaju predvidljive, validirane podatke  
3. **Agenti s jednom odgovornošću** — Dizajniranje fokusiranih agenata koji dobro obavljaju jednu zadaću

Svaki ćemo obrazac primijeniti na scenarij **preporučitelja turističkih destinacija**, postupno gradeći sustav koji može predlagati destinacije, provjeravati dostupnost i rješavati logistiku.


## Instalacija


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Uzorak 1: Jasne Upute za Agenta

Najjači uzorak je također i najjednostavniji: pisanje jasnih, detaljnih uputa za vašeg agenta.

Dobre upute definiraju:
- **Tko** je agent (persona i ton)
- **Što** treba raditi (odgovornosti korak po korak)
- **Kako** se treba ponašati (ograničenja i stil)

Ispod kreiramo agenta turističkog conciergea s eksplicitnim uputama koje oblikuju svaki odgovor koji daje.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Obrasac 2: Strukturirani Izlaz s Pydantic Modelima

Tekst slobodnog oblika koristan je za razgovor, ali sustavi u nastavku trebaju strukturirane podatke.
Povezujući **Pydantic modele** s **funkcijom alata**, možemo:

- Definirati točan obrazac za izlaz agenta
- Automatski provjeravati valjanost odgovora
- Pouzdano integrirati rezultate agenta u logiku aplikacije

Također uvodimo alat koji vraća podatke o odredištu kako bi agent svoje preporuke temeljio na stvarnim podacima.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Obrasac 3: Agent s Jednom Odgovornošću

Složeni zadaci imaju koristi od podjele rada na više fokusiranih agenata, od kojih svaki ima jednu odgovornost:

- **Stručnjak za odredište** koji zna o mjestima i dostupnosti
- **Planer logistike** koji upravlja letovima, hotelima i itinererima

Ovo odražava načelo softverskog inženjerstva *odvajanja briga* — svaki agent je lakše testirati, održavati i poboljšavati neovisno.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Sažetak

U ovoj lekciji primijenili smo tri obrazca agentnog dizajna na scenarij preporučitelja putovanja:

| Obrazac | Ključna ideja | Prednost |
|---|---|---|
| **Jasne upute** | Definirati osobu, odgovornosti i ograničenja unaprijed | Dosljedno, u skladu s brendom ponašanje agenta |
| **Strukturirani izlaz** | Koristiti Pydantic modele kao format odgovora | Validirani, strojno čitljivi rezultati |
| **Pojedinačna odgovornost** | Svakom agentu dati jedan usmjeren zadatak | Lakše testiranje, održavanje i sastavljanje |

Ovi se obrasci prirodno nadopunjuju — možete kombinirati jasne upute sa strukturiranim izlazom unutar agenta s pojedinačnom odgovornošću za izgradnju robusnih sustava spremnih za proizvodnju.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
